# RQ2 --- Efficiency

In [1]:
import sys

sys.path.insert(0, "..")
import pandas as pd
from experiments._loader import load_all_results, success_only
from experiments._analysis import setup_matplotlib

setup_matplotlib()
df_all = load_all_results(include_baseline_fail=True)
df = success_only(df_all)
df["total_budget"] = (df["img_budget_used"] + df["txt_budget_used"]).clip(upper=df["budget_max"])
df["budget_utilization"] = df["total_budget"] / df["budget_max"]
df["total_evaluations"] = df["generations_completed"] * 100  # Number of individuals in generation

counts = df["model"].value_counts()
df = df[df["model"].isin(counts[counts >= 15].index)]

In [2]:
METRIC_COLS = [
    "img_budget_used",
    "txt_budget_used",
    "total_budget",
    "total_evaluations",
    "runtime",
]

SCENES = [
    ("MC", "multi"),
    ("SC-MI", "single/multi"),
    ("SC-SI", "single/solo"),
    ("Driving", "udacity"),
]


def format_metric(series: pd.Series) -> str:
    values = series.dropna()

    if values.empty:
        return "---"

    mean = values.mean()
    std = values.std()
    std = 0.0 if pd.isna(std) else std

    return f"${mean:.2f} \\pm {std:.2f}$"


records = []
for model in sorted(df["model"].dropna().unique()):
    for scene_label, obj_category in SCENES:
        scene_mask = (df["model"] == model) & (df["obj_category"] == obj_category)

        for genome_mode in ["multi", "image", "text"]:
            sub = df.loc[scene_mask & (df["genome_mode"] == genome_mode)]

            records.append(
                {
                    "model": model,
                    "scene": scene_label,
                    "genome_mode": genome_mode,
                    "img_budget": format_metric(sub["img_budget_used"]),
                    "txt_budget": format_metric(sub["txt_budget_used"]),
                    "total_budget": format_metric(sub["total_budget"]),
                    "total_evaluations": format_metric(sub["total_evaluations"]),
                    "runtime": format_metric(sub["runtime"]),
                }
            )

efficiency_df = pd.DataFrame(records)

In [3]:
efficiency_df.to_csv("efficiency.csv")


In [4]:
from experiments._analysis import MetricTable

In [5]:
MODEL_MAPPING = {
    "qwen": "Qwen3-VL",
    "kimi": "Kimi-VL",
    "intern": "InternVL3.5",
    "gemma": "Gemma3",
    "deepseek": "DeepseekVL2",
    "nemotron": "Nemotron3-NO",
}
efficiency_df = pd.read_csv("efficiency.csv")

efficiency = MetricTable(
    df=efficiency_df,
    title="Efficiency",
    model_mapping={k: v for k, v in MODEL_MAPPING.items() if k in efficiency_df["model"].unique()},
    scene_mapping={d: d for d in efficiency_df["scene"].unique()},
    metric_mapping={
        "img_budget": r"\faImage\ Budget $\downarrow$",
        "txt_budget": r"\faFont\ Budget $\downarrow$",
        "total_budget": r"$\sum$ Budget $\downarrow$",
        "total_evaluations": r"\# SUT Eval $\downarrow$",
        "runtime": r"Runtime (sec) $\downarrow$",
    },
)

In [6]:
print(efficiency)

\begin{tabular}{llccccc}
\toprule
Model & Scene & \faImage\ Budget $\downarrow$ & \faFont\ Budget $\downarrow$ & $\sum$ Budget $\downarrow$ & \# SUT Eval $\downarrow$ & Runtime (sec) $\downarrow$ \\
\midrule
\multirow{12}{*}{Qwen3-VL} & \multirow{3}{*}{MC} & $0.20 \pm 0.15$ & $0.41 \pm 0.16$ & $0.60 \pm 0.22$ & $916.25 \pm 2159.84$ & $33.93 \pm 62.95$ \\
 &  & $0.22 \pm 0.14$ & $0.00 \pm 0.00$ & $0.22 \pm 0.14$ & $6006.25 \pm 4697.81$ & $193.70 \pm 162.88$ \\
 &  & $0.00 \pm 0.00$ & $0.44 \pm 0.15$ & $0.44 \pm 0.15$ & $1116.25 \pm 2387.44$ & $30.00 \pm 49.84$ \\
\cmidrule(lr){2-7}
 & \multirow{3}{*}{SC-MI} & $0.19 \pm 0.15$ & $0.44 \pm 0.16$ & $0.63 \pm 0.22$ & $1624.47 \pm 2726.40$ & $61.04 \pm 92.72$ \\
 &  & $0.21 \pm 0.12$ & $0.00 \pm 0.00$ & $0.21 \pm 0.12$ & $5992.55 \pm 4680.25$ & $158.31 \pm 137.09$ \\
 &  & $0.00 \pm 0.00$ & $0.51 \pm 0.20$ & $0.51 \pm 0.20$ & $2747.87 \pm 3832.14$ & $54.81 \pm 79.86$ \\
\cmidrule(lr){2-7}
 & \multirow{3}{*}{SC-SI} & $0.14 \pm 0.14$ & $0.53 \p